# 第一天：Agent 智能体 Skill

这个 Notebook 演示 V2 如何根据用户需求按需加载标准 `SKILL.md`。

学生可以对比普通问答、英语剧情闯关和错题整理，观察 Agent 如何为不同需求选择不同 Skill。

## 1-1. 导入统一接口

模型、Skill 发现和 Agent 调用逻辑全部来自 `src/`。

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd()
if project_root.name == "teacher":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.artifacts import discover_skills, read_markdown
from src.facade import invoke

## 1-2. 查看标准 SKILL.md

Agent 启动时只看到 `name` 和 `description`。

完整步骤要等到需求匹配并调用 `load_skill` 后才进入模型上下文。

In [ ]:
skills = discover_skills()
for skill in skills:
    print(f"Skill：{skill.name}")
    print(f"触发说明：{skill.description}")
    print(f"文件：{skill.path.relative_to(project_root)}")
    print()
    print(read_markdown(skill.path))

## 1-3. 搭建按需加载 Skill 的 Agent

- 输入普通英语知识问题，观察 Agent 是否保持普通问答而不加载 Skill。

- 输入英语闯关请求，观察 Agent 是否加载 `english-quest` 并通过对话历史延续游戏。

- 在 Streamlit 课程页中演示同一句请求，观察应用如何自动进入专属闯关页面，并用 Skill 中的 `scripts/` 和 `assets/` 呈现互动状态。

- 输入整理错题请求或 `student/mistakes/inbox/` 内的文件路径，观察 Agent 是否加载 `sorting-out-mistakes`。

输入 `/exit` 可以结束对话。

In [ ]:
def run_v2_dialogue():
    history = []
    tool_calls = []
    print("🧰 Skill Agent 已就绪。")

    while True:
        student_message = input("学生：").strip()
        if student_message.lower() in {"/exit", "exit", "退出"}:
            print("期待下次交流。")
            return {"history": history, "tool_calls": tool_calls}
        if not student_message:
            print("输入不能为空，请重新输入。")
            continue

        turn_result = invoke("V2", student_message, history=history)
        if turn_result["error"]:
            raise RuntimeError(turn_result["error"])

        loaded_skills = [
            call.get("args", {}).get("skill_name")
            for call in turn_result["tool_calls"]
            if call.get("name") == "load_skill"
        ]
        for skill_name in loaded_skills:
            print(f"🔧 Agent 按需加载了 Skill：{skill_name}")

        loaded_files = [
            call.get("args", {}).get("path")
            for call in turn_result["tool_calls"]
            if call.get("name") == "load_mistake_file"
        ]
        for file_path in loaded_files:
            print(f"📖 Agent 已读取错题文件：{file_path}")

        saved_mistakes = [
            call
            for call in turn_result["tool_calls"]
            if call.get("name") == "save_mistake"
        ]
        for _ in saved_mistakes:
            print("💾 Agent 已调用 save_mistake 写入错题。")

        tool_calls.extend(turn_result["tool_calls"])
        history.extend([
            {"role": "user", "content": student_message},
            {"role": "assistant", "content": turn_result["text"]},
        ])
        print(f"Carl教练：{turn_result['text']}")

## 1-4. 三次对比真实对话

在同一个对话中依次输入：

1. `什么是现在完成时？`
2. `我们玩一个侦探闯关游戏练现在完成时。`
3. 游戏中答错后输入 `把刚才答错的题整理进错题本。`

观察第一次不加载 Skill，第二次加载 `english-quest` 并进入专属页面，第三次切换到 `sorting-out-mistakes`。

In [ ]:
v2_session = run_v2_dialogue()